In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [0]:
df = spark.read.format("csv").option("header", "true") \
.option("inferSchema", "true") \
.option('sep',';') \
.load("/Volumes/workspace/raw/transacciones/superstore2.csv")

In [0]:
#Creamos el esquema bronze si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
#Creamos la tabla y guardamos la información
df.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .option('delta.columnMapping.mode', 'name') \
    .saveAsTable('workspace.bronze.superstore')

In [0]:
df.count()

In [0]:
display(df.head(10))

In [0]:
# Creamos el dataframe de control con esquema explícito
schema = StructType([
    StructField("nombre_capa", StringType(), True),
    StructField("nombre_tabla", StringType(), True),
    StructField("cnt_registros", IntegerType(), True)
])

df_control = spark.createDataFrame([
    ("bronze", "superstore", int(df.count()))
], schema) \
    .withColumn("fecha_actualizacion", current_timestamp())


spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ctl")

# Creamos la tabla de control si no existe
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.ctl.control_procesos (
    nombre_capa STRING,
    nombre_tabla STRING,
    cnt_registros INT,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
""")

# Insertamos el registro en la tabla de control
df_control.write.format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('workspace.ctl.control_procesos')


display(df_control)

In [0]:
df_control